# 🍐 Little Fig — Workspace
### Unified Model Training & Dashboard

Train LLMs on any hardware, write memories into weights, and monitor everything from a built-in web dashboard — all from one notebook.

| Research Finding | Improvement |
|---|---|
| FigMeZO (inverse error shaping) | −18.6% loss vs standard MeZO |
| Sensitivity-guided LISA | −10% loss vs random layer selection |
| GPU training speed | 7× faster than BnB NF4 QLoRA |
| Quantization quality | Wins 156/156 TinyLlama layers vs NF4 |

**Author:** 0xticketguy / Harboria Labs | **License:** Mixed; see NOTICE.md

[![GitHub](https://img.shields.io/badge/GitHub-littlefig-black)](https://github.com/Harboria-Labs/littlefig)

**Notebook layout:**
1. Setup & Installation
2. Quick Start — Fine-tune TinyLlama
3. Launch the Dashboard (UI)
4. Execute Training (with live monitoring)
5. Memory Fabric — weight-space memory
6. FigMeZO — backward-pass-free training
7. Benchmark Results


## 1. Setup & Installation


In [ ]:
#@title 🛠️ Step 1: Environment Setup
# Note: using the PEP 508 'req @ url' syntax instead of an egg fragment
# (the egg fragment form throws a pip 25 deprecation warning)
!pip install -q torch
!pip install -q "little-fig[train] @ git+https://github.com/Harboria-Labs/littlefig.git"

import torch
import little_fig
print(f'✅ Ready | PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name()}')


## 2. Quick Start — Fine-tune TinyLlama in 5 Minutes


In [ ]:
#@title 🚀 Step 2: Initialize Model + Trainer
from little_fig.engine import FigModel, FigTrainer, FigTrainingConfig
from little_fig.engine.tier import TrainingTier

# Load TinyLlama with FigQuant INT4 + LoRA
model = FigModel.from_pretrained(
    'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    lora_r=16,
    lora_alpha=32,
    shared_codebook=True,  # 5× faster loading
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

# Configure training
config = FigTrainingConfig(
    num_epochs=1,
    learning_rate=2e-4,
    max_seq_length=256,  # shorter for Colab speed
    batch_size=2,
    gradient_accumulation_steps=4,
    logging_steps=5,
    use_packing=True,
)

trainer = FigTrainer(model, config)
trainer.load_dataset('tatsu-lab/alpaca', max_samples=200)

print("\n✅ Model and Trainer initialized.")
print("   Next: launch the dashboard (Section 3) before running trainer.train() (Section 4),")
print("   so you can watch training happen live.")


## 3. Launch the Dashboard (UI)

The UI lives in `little_fig.web.server` — a Flask app that reads live state from your `trainer` object. It ships as part of the package (`little_fig/web/server.py` + `little_fig/web/static/index.html`), so no separate install is needed.

Run **both** cells below. The first starts the server in a background thread. The second gives you two ways to view it: an inline iframe, or an external tab via Colab's port proxy. You only need one — pick whichever you prefer.


In [ ]:
#@title 🖥️ Step 3a: Start the Dashboard Server
import threading
from little_fig.web import server

def run_dashboard():
    server.app.config['TRAINER'] = trainer
    server.app.run(host='0.0.0.0', port=5000)

dashboard_thread = threading.Thread(target=run_dashboard, daemon=True)
dashboard_thread.start()

import time; time.sleep(2)  # give Flask a moment to bind
print('✅ Dashboard server running on port 5000.')
print('   Run the next cell to view it.')


In [ ]:
#@title 🖥️ Step 3b: View the Dashboard

# ── Option A: Inline iframe (view directly in this notebook) ──────────────
from google.colab import output
print("--- INLINE DASHBOARD ---")
output.serve_kernel_port_as_iframe(5000, height='600')

# ── Option B: External link (full browser tab) ─────────────────────────────
from google.colab.output import eval_js
proxy_url = eval_js("google.colab.kernel.proxyPort(5000)")
print(f"\n--- EXTERNAL LINK ---\nClick here for full browser view:\n{proxy_url}")


**Not on Colab?** (Kaggle, local Jupyter, plain terminal) — the iframe/proxy calls above are Colab-specific and will fail elsewhere. Run the server directly instead:

```python
from little_fig.web import server
server.app.config['TRAINER'] = trainer
server.app.run(host='0.0.0.0', port=5000)
# Then open http://localhost:5000 in your browser
```

**Using ngrok instead** (e.g. to share the dashboard outside your own machine):

```python
!pip install -q pyngrok
from pyngrok import ngrok
public_url = ngrok.connect(5000)
print(public_url)
```


## 4. Execute Training

With the dashboard open above, run training below — progress updates live in the UI.


In [ ]:
#@title 🏃 Step 4: Execute Training
# Monitor progress in the dashboard above while this runs
trainer.train()

# Save (only ~5MB for the adapter)
model.save_adapter('./my_adapter')
print('✅ Adapter saved to ./my_adapter')


## 5. Memory Fabric — The Model Remembers

Memory lives IN the model weights. No external database. No RAG.


In [ ]:
# Load with Memory Fabric
model = FigModel.from_pretrained(
    'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    lora_r=16,
    memory_fabric=True,
    shared_codebook=True,
)

# Write memories INTO the weights
r1 = model.write_memory('personal', 'User prefers Python for backend work.')
r2 = model.write_memory('wiki', 'Speed of light is 299,792,458 m/s.')
r3 = model.write_memory('schedule', 'Team standup every day at 9:15am.')

print(f'Memory written in {r1["time_ms"]:.0f}ms')
print(f'\nMemory confidence per namespace:')
for ns, info in model.memory_confidence().items():
    if info['mean_magnitude'] > 0:
        print(f'  {ns}: {info["mean_magnitude"]:.4f}')


## 6. FigMeZO — Train Without Backward Passes

Original research: −18.6% loss vs standard MeZO. Uses only forward passes — fits in inference-level memory.


In [ ]:
from little_fig.engine.figmezo import FigMeZO, FigMeZOConfig

# MeZO: gradient-free training (only forward passes!)
optimizer = FigMeZO(model.model, FigMeZOConfig(
    learning_rate=1e-5,
    epsilon=1e-3,
    shaping_strength=-0.3,  # Negative = our novel inverse shaping
))

# Each step uses 2 forward passes, 0 backward passes
import torch
model.model.eval()
for step in range(5):
    ids = torch.randint(0, 32000, (1, 32))
    if torch.cuda.is_available(): ids = ids.cuda()
    loss = optimizer.step(lambda: model(input_ids=ids, labels=ids).loss)
    print(f'  Step {step}: loss={loss:.4f}')


## 7. Benchmark Results

All results validated on Tesla T4 GPU with TinyLlama 1.1B.

### Quantization Quality (156 layers)

| Method | MSE | Cosine | Wins |
|---|---|---|---|
| **FigQuant** | **5.64e-6** | **0.9956** | **156/156** |
| NF4 (QLoRA) | 5.97e-6 | 0.9953 | 0/156 |

### Training Speed

| Method | Loss | Time | Speed |
|---|---|---|---|
| FP16 LoRA | 0.2252 | 1309s | 1× |
| BnB NF4 | 0.2399 | 1423s | 0.9× |
| **FigQuant** | **0.2475** | **184s** | **7×** |

---

*Built by 0xticketguy / Harboria Labs*
*License: Mixed; see NOTICE.md*
